# ShopBot v2 — 67 sản phẩm và shop mô phỏng một năm

Hai loại dữ liệu dùng chung chatbot, không dùng chung một loss huấn luyện.
1. Embedding: E5 multilingual small, chạy GPU, so baseline với candidate.
2. Dữ liệu số: không train thuộc số. Huấn luyện riêng dự báo nhu cầu một ngày,
   so naive cùng thứ tuần trước với gradient boosting; chọn bằng validation.
3. Kev: định tuyến public, dùng notebook Kev riêng sau khi duyệt tập nhãn v2.
   Quyền admin luôn kiểm tra ở mã ứng dụng, không do Kev quyết định.

Không upload `.env`, khách hàng thật, giao dịch thật, báo cáo bản quyền đầy đủ,
đáp án sự kiện cài sẵn hoặc nhu cầu ẩn. Chỉ có thông số công khai và dữ liệu tổng hợp giả lập.


In [ ]:
import base64,json,zlib,sys,subprocess
from pathlib import Path
ROOT=Path('/content/shopbot_tutor_v2');ROOT.mkdir(parents=True,exist_ok=True)
FILES=json.loads(zlib.decompress(base64.b64decode('...')))
for name,text in FILES.items():
    path=(ROOT/name).resolve()
    assert path.is_relative_to(ROOT.resolve())
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(text,encoding='utf-8')
print(json.dumps(json.loads((ROOT/'data/manifest.json').read_text()),ensure_ascii=False,indent=2))


In [ ]:
import torch
assert torch.cuda.is_available(),'Chọn runtime GPU trước khi train retriever.'
print('GPU:',torch.cuda.get_device_name(0))
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.51,<6','huggingface-hub>=0.30','sentencepiece','scikit-learn','pandas'],check=True)
def run_live(command):
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        for line in process.stdout: print(line,end='',flush=True)
        if process.wait(): raise RuntimeError('Training process failed; inspect output above.')


## Retriever

Tập v1 giữ nguyên group/split; v2 thêm nguồn và kế hoạch. Câu hỏi khuôn mẫu chỉ là pilot.
Chỉ dùng mẫu supported để train; chỉ chọn trọng số mới nếu validation recall@1 tăng.
Kết quả này chưa đo ACL, thời gian hiệu lực, multi-hop hay chất lượng câu trả lời tự do.


In [ ]:
run_live([sys.executable,str(ROOT/'retriever_train.py'),'--root',str(ROOT),'--epochs','3'])


## Dự báo riêng

Tám tháng train, hai tháng validation, hai tháng test. Dùng lags quá khứ; test walk-forward
một ngày với dữ liệu ngày trước được quan sát. Không phải dự báo cố định trước 60 ngày.
Không khẳng định deep learning tốt hơn: một năm giả lập không đủ chứng minh điều đó.


In [ ]:
run_live([sys.executable,str(ROOT/'forecast_train.py'),'--root',str(ROOT)])


## Lưu checkpoint và báo cáo, chưa thay model production

In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/shopbot_v2_outputs','zip',ROOT/'outputs')
print('Artifact:',archive)
files.download(archive)
